# Project 4 — HR Attrition Analysis

Investigate aggregate attrition patterns and practice responsible reporting. The dataset is entirely synthetic and must not be used to make real employment decisions.

## Load and quality-check HR data

This dataset is synthetic and intended only for analytical practice. Check missingness, valid ranges, unique employee keys, and the distribution of the outcome before analysis. Never use this example to make real employment decisions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path  # Import path utilities for local CSV loading.
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")  # Resolve the data folder.
df = pd.read_csv(DATA_DIR / "employees.csv")  # Load synthetic employee-level data.
print("Shape:", df.shape)  # Show dataset dimensions.
print(df.head())  # Preview example records.
print("Duplicate employee IDs:", df["employee_id"].duplicated().sum())  # Check the employee key.
print("\nMissing values:\n", df.isna().sum())  # Check missing values.
print("\nAttrition distribution:\n", df["attrition"].value_counts(normalize=True).round(3))  # Display the outcome distribution.

## Calculate overall and group attrition rates

Attrition rate is often calculated as separations divided by an agreed employee base over a period. This supplied dataset is a cross-sectional teaching dataset, so the `attrition` field should be interpreted as a recorded outcome—not a fully specified monthly HR rate.

Compare group rates descriptively and display denominators. Different group sizes can make raw attrition counts misleading.

In [ ]:
overall_attrition = (df["attrition"] == "Yes").mean()  # Calculate the overall proportion labeled as attrition.
dept_report = df.groupby("department").agg(  # Aggregate at department level.
    employees=("employee_id", "nunique"),  # Count employees in each group.
    attrition_count=("attrition", lambda s: (s == "Yes").sum()),  # Count employees marked as attrited.
)  # Finish group aggregation.
dept_report["attrition_rate"] = dept_report["attrition_count"] / dept_report["employees"]  # Calculate the group attrition proportion.
print(f"Overall recorded attrition proportion: {overall_attrition:.1%}")  # Display overall outcome share.
print(dept_report.sort_values("attrition_rate", ascending=False).round(3))  # Rank departments by observed rate.

## Compare descriptive patterns

Look at how outcome rates vary with overtime, satisfaction, tenure, and business travel. Such differences can suggest questions for further investigation; they do not establish why an employee leaves.

Avoid interpreting sensitive or protected attributes as reasons to take action against individuals. Prefer process-level interventions, aggregation, fairness review, and voluntary employee feedback.

In [ ]:
overtime_report = df.groupby("overtime").agg(  # Compare records with and without overtime.
    employees=("employee_id", "nunique"),  # Count employees in each overtime group.
    attrition_rate=("attrition", lambda s: (s == "Yes").mean()),  # Calculate outcome share within each group.
)  # Finish overtime summary.
satisfaction_report = df.groupby("job_satisfaction_1_to_4")["attrition"].apply(lambda s: (s == "Yes").mean())  # Calculate attrition share at each satisfaction score.
df["tenure_band"] = pd.cut(df["years_at_company"], bins=[0, 1, 3, 5, 10, 40], labels=["<1", "1-3", "3-5", "5-10", "10+"])  # Group tenure into broad bands.
tenure_report = df.groupby("tenure_band", observed=False)["attrition"].apply(lambda s: (s == "Yes").mean())  # Calculate outcome share by tenure band.
print("By overtime:\n", overtime_report.round(3))  # Display overtime comparisons with denominators.
print("\nBy satisfaction score:\n", satisfaction_report.round(3))  # Display satisfaction-group patterns.
print("\nBy tenure band:\n", tenure_report.round(3))  # Display tenure-group patterns.

## Visualize department rates

A clear chart makes group differences easier to compare. Show rates rather than counts when department sizes differ, and pair the chart with denominators so small groups do not appear more certain than they are.

In [ ]:
dept_report = dept_report.sort_values("attrition_rate", ascending=False)  # Order the departments by recorded attrition proportion.
fig, ax = plt.subplots(figsize=(9, 4))  # Create the bar-chart figure.
ax.bar(dept_report.index, dept_report["attrition_rate"])  # Plot proportion by department.
ax.set_title("Recorded Attrition Proportion by Department")  # Add a descriptive title.
ax.set_xlabel("Department")  # Label the categories.
ax.set_ylabel("Proportion marked Yes")  # Label the rate axis.
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))  # Display the y-axis in percentages.
plt.xticks(rotation=25, ha="right")  # Improve category-label readability.
plt.tight_layout()  # Adjust the chart margins.
plt.show()  # Display the chart.
print(dept_report[["employees", "attrition_count", "attrition_rate"]].round(3))  # Show group sizes alongside rates.

## Recommend responsible next steps

The goal is to identify questions for responsible follow-up, not to predict who should be dismissed or denied opportunities. Potential actions include reviewing workload, surveying teams, investigating process differences, and evaluating voluntary retention programs.

Before publishing, document the synthetic nature of the data, missing context, sample-size limitations, group definitions, and potential biases.

In [ ]:
findings = {  # Prepare a responsible reporting checklist.
    "Descriptive signal": "Summarize the largest observed group differences with counts and denominators.",  # Describe patterns without claiming causes.
    "Validation": "Ask HR and team leaders to review process context and data definitions.",  # Validate data and interpretation.
    "Potential intervention": "Pilot workload, scheduling, or manager-support improvements at an aggregate level.",  # Focus on process changes.
    "Evaluation": "Measure future aggregate retention and employee feedback against a comparison where feasible.",  # Define an evaluation plan.
    "Safeguard": "Do not use this synthetic analysis to score, rank, or make employment decisions about individuals.",  # State a clear limitation.
}  # Finish the action checklist.
for heading, statement in findings.items():  # Iterate through the recommendations.
    print(f"{heading}: {statement}")  # Display each recommendation.
print("Portfolio deliverable: aggregate dashboard, caveats, and a responsible intervention plan.")  # Define what to publish.